In [1]:
#[CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: Bautista TUPM-26-3506
Seed Digit: 6
ID Sum: 22
Name Length: 8
Random Seed: 263506
Verification Value: 182


In [3]:
#[CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measuremnent_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measuremnent_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:" , WORK_DIR.resolve())
print("All datasets files verified.")

Workspace: C:\Users\jacob\Documents\Comprog\Lab7_NumPy_Pandas_Bautista\Lab7_NumPy_Pandas_Additional_Exercises\vectorized_measurement_quality_analysis
All datasets files verified.


In [5]:
# [CELL 3] MISSING VALUES AND NUMPY STATISTICS
raw_measurements = np.genfromtxt(measuremnent_path, dtype=float)
assert raw_measurements.ndim == 2

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
    missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
    missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

raw_usable = raw_measurements[:, usable_indices].copy()

feature_means = np.nanmean(raw_usable, axis=0)
feature_sds = np.nanstd(raw_usable, axis=0, ddof=1)

print("Total Missing Values:", missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Retained Features:", len(usable_indices))
print("First Feature Mean:", f"{feature_means[0]:.4f}")
print("First Feature Sample SD:", f"{feature_sds[0]:.4f}")

Total Missing Values: 41951
Entirely Missing Features: 0
Constant Features: 116
Retained Features: 474
First Feature Mean: 3014.4529
First Feature Sample SD: 73.6218


In [14]:
#[CELL 4] REQUIREMENT 2

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    len(usable_indices),
    size= 5 + (SEED_DIGIT % 3),
    replace=False
)

selected_features = usable_indices[selected_indices]

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

selected_feature_names = [
    feature_names[i]
    for i in selected_indices
]

selected_data = raw_measurements[
    :,
    selected_indices
]

print("Feature Count:",
    size := len(selected_indices))

print("\nSelected Feature Indices:")
print(selected_indices)

print("\nSelected Features:")
print(selected_feature_names)

print("\nSelected Data Shape:")
print(selected_data.shape)

Feature Count: 5

Selected Feature Indices:
[323 401 465 101 464]

Selected Features:
['Feature_323', 'Feature_401', 'Feature_465', 'Feature_101', 'Feature_464']

Selected Data Shape:
(1567, 5)


In [18]:
# [CELL 5] MEAN MEDIAN SAMPLE SD AND MISSING PERCENTAGE

feature_means = np.nanmean(
    selected_data,
    axis=0
) #feature means ignoring NaN values

feature_medians = np.nanmedian(
    selected_data,
    axis=0
) #feature medians ignoring NaN values

feature_sds = np.nanstd(
    selected_data,
    axis=0,
    ddof=1
) #sample standard deviations ignoring NaN values

# percentage of missing values for each feature
missing_counts = np.isnan(
    selected_data
).sum(axis=0)

missing_percentages = (
    missing_counts
    / selected_data.shape[0]
    * 100
)

#dataframe to store the measurement quality metrics
measurement_quality = pd.DataFrame({
    "Feature": selected_feature_names,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_SD": feature_sds,
    "Missing_Percentage": missing_percentages
})

print(measurement_quality.round(4))

       Feature    Mean  Median  Sample_SD  Missing_Percentage
0  Feature_323  5.3934  5.4588     2.5189              0.0638
1  Feature_401  0.0000  0.0000     0.0000              0.5743
2  Feature_465  0.0000  0.0000     0.0000              0.2553
3  Feature_101 -0.0000  0.0000     0.0002              0.3829
4  Feature_464  0.0000  0.0000     0.0000              0.2553


In [23]:
# [CELL 6] ROBUST STANDARDIZATION

#MEDIANS 
medians = np.nanmedian(
    selected_data,
    axis=0
)

#Absolute deviations from the median
absolute_deviations = np.abs(
    selected_data - medians
)

#MAD (Median Absolute Deviation)
mad = np.nanmedian(
    absolute_deviations,
    axis=0
)

safe_mad = np.where(
    mad > 0,
    mad,
    1.0
)

#Robust Standardization
robust_z = (
    selected_data - medians
) / safe_mad

# Display the results
print("Feature Medians:")
print(medians)

print("\nMAD Values:")
print(mad)

print("\nRobust Z Shape:")
print(robust_z.shape)

print(
    "\nFeatures With Zero MAD:",
    np.count_nonzero(mad == 0)
)

for feature, value in zip(
    selected_feature_names,
    mad
):
    print(
        f'\n {feature}: MAD = {round(value, 4)}'
    )

Feature Medians:
[5.4588 0.     0.     0.     0.    ]

MAD Values:
[1.0041 0.     0.     0.0001 0.    ]

Robust Z Shape:
(1567, 5)

Features With Zero MAD: 3

 Feature_323: MAD = 1.0041

 Feature_401: MAD = 0.0

 Feature_465: MAD = 0.0

 Feature_101: MAD = 0.0001

 Feature_464: MAD = 0.0


In [24]:
# [CELL 7] IDENTIFICATION
flags = np.abs(robust_z) > 3.5

flagged_counts = np.count_nonzero(
    flags,
    axis=0
)

flagged_percentages = (
    flagged_counts
    / selected_data.shape[0]
    * 100
)

total_flagged = np.count_nonzero(flags)

measurement_quality[
    "Flagged_Count"
] = flagged_counts

measurement_quality[
    "Flagged_Percentage"
] = flagged_percentages

print(
    measurement_quality[
        [
            "Feature",
            "Flagged_Count",
            "Flagged_Percentage"
        ]
    ].round(4)
)

print(
    "Total Flagged Observations:",
    total_flagged
)

       Feature  Flagged_Count  Flagged_Percentage
0  Feature_323              6              0.3829
1  Feature_401              0              0.0000
2  Feature_465              0              0.0000
3  Feature_101            111              7.0836
4  Feature_464              0              0.0000
Total Flagged Observations: 117


In [ ]:
#[CELL 8] ASSESMENT OUTPUT

measurement_quality = (
    measurement_quality
    .sort_values(
        by="Flagged_Percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

highest_flagged_feature = (
    measurement_quality.iloc[0]["Feature"]
)

highest_flagged_percentage = (
    measurement_quality.iloc[0]["Flagged_Percentage"]
)

highest_missing_percentage = (
    measurement_quality[
        "Missing_Percentage"
    ].max()
)

quality_path = (
    OUTPUT_DIR /
    "measurement_quality.csv"
)

measurement_quality.to_csv(
    quality_path,
    index=False
)

loaded_quality = pd.read_csv(
    quality_path
)

pd.testing.assert_frame_equal(
    loaded_quality,
    measurement_quality,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10
)

verified = True

print("Selected Features:")
print(selected_feature_names)

print(
    "\nHighest Missing Percentage:",
    round(highest_missing_percentage, 4)
)

print(
    "Feature with Highest Flagged Percentage:",
    highest_flagged_feature
)

print(
    "Total Flagged Observations:",
    total_flagged
)

print(
    "Vectorized Results Verified:",
    verified
)

print("\nExported File:")
print(quality_path.name)

Selected Features:
['Feature_323', 'Feature_401', 'Feature_465', 'Feature_101', 'Feature_464']

Highest Missing Percentage: 0.5743
Feature with Highest Flagged Percentage: Feature_101
Total Flagged Observations: 117
Vectorized Results Verified: True

Exported File:
measurement_quality.csv
